## Hospital Resource Management Bot

The AI must:

    Provide resource management assistance only when sufficient hospital context is available

    Refuse or respond with "Unable to determine" when critical resource information is missing

    Ask clarification questions instead of assuming or guessing hospital resource details


## Step 1: Add Guardrails to Static Context

We now enforce *when the AI should refuse to answer*.

In [9]:
SYSTEM_CONTEXT_GUARDED = """
You are a Hospital Resource Management Bot.

Rules:
- Do not assume missing hospital resource information
- If required data is missing, ask a clarification question
- If resource availability cannot be determined, respond with "Unable to determine"
- Never invent or assume bed, equipment, staff, or patient information
- Follow hospital policies and user access permissions strictly
- Do not provide medical diagnosis or treatment advice
- Protect patient and hospital information
- Be polite and professional
"""

## Step 2: Incomplete User Context (Simulated Real-World Issue)

Here, we intentionally remove critical information.

In [10]:
user_query = "Can you check and manage the available hospital resources?"

user_profile_incomplete = {
    "role": "hospital_staff",
    "department": "Emergency"
    # Missing resource type
    # Missing resource status
    # Missing specific resource requirement
    # Missing priority
}

## Step 3: Assemble Context with Missing Information

Notice: we DO NOT fill missing values.

In [11]:
HOSPITAL_RESOURCE_POLICY = """
Hospital Resource Management Policy:
- Hospital resources must be allocated only to authorized staff
- Beds marked as occupied or under maintenance cannot be assigned
- Medical equipment marked as unavailable or under maintenance cannot be allocated
- Staff assignments must consider department and availability
- Emergency requests receive priority for critical resources
- Critical resource shortages must be escalated to the hospital administrator
- Patient and hospital information must remain confidential
"""

user_query = "Can you manage the available hospital resources?"

user_profile_incomplete = {
    "role": "hospital_staff",
    "department": "Emergency"
    # Missing resource type
    # Missing resource status
    # Missing specific requirement
    # Missing priority
}


In [12]:
final_prompt_incomplete = f"""
{SYSTEM_CONTEXT_GUARDED}

Hospital Resource Management Policy:
{HOSPITAL_RESOURCE_POLICY}

User Profile:
- Role: {user_profile_incomplete['role']}
- Department: {user_profile_incomplete['department']}
- Resource Type: Missing
- Resource Status: Missing
- Requirement: Missing
- Priority: Missing

User Question:
{user_query}
"""

In [13]:
from google.colab import userdata
from openai import OpenAI

# 1. Load API key from Colab Secrets
MY_API_KEY = userdata.get("api_key")

# 2. Create OpenAI-compatible client
client = OpenAI(
    api_key=MY_API_KEY,
    base_url="https://nexusapi.navigatelabs.ai"
)

# 3. Send request to the model
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {
            "role": "system",
            "content": SYSTEM_CONTEXT_GUARDED
        },
        {
            "role": "user",
            "content": final_prompt_incomplete
        }
    ]
)

# 4. Print model response
print(response.choices[0].message.content)

I can help manage hospital resources. To assist you, please provide more details about your request.

What specific resource are you looking for (e.g., a bed, a piece of equipment, staff)? What is your requirement?


## Step 4: Conflicting Context

Real systems often receive contradictory data.

In [14]:
user_profile_conflict = {
    "role": "nurse",
    "department": "ICU",
    "resource_type": "ventilator",
    "resource_status": "under maintenance",  # Conflicts with resource policy
    "request": "Assign this ventilator to a patient"
}

In [15]:
final_prompt_conflict = f"""
{SYSTEM_CONTEXT_GUARDED}

Hospital Resource Management Policy:
{HOSPITAL_RESOURCE_POLICY}

User Profile:
- Role: {user_profile_conflict['role']}
- Department: {user_profile_conflict['department']}
- Resource Type: {user_profile_conflict['resource_type']}
- Resource Status: {user_profile_conflict['resource_status']}
- Request: {user_profile_conflict['request']}

User Question:
{user_query}
"""

In [16]:
response = client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "system", "content": SYSTEM_CONTEXT_GUARDED},
        {"role": "user", "content": final_prompt_conflict}
    ]
)

print(response.choices[0].message.content)

I can assist with managing available hospital resources according to hospital policies.

However, regarding your request to assign the ventilator: The hospital policy states that medical equipment marked as 'under maintenance' cannot be allocated. Therefore, I am unable to assign the ventilator that is currently under maintenance to a patient.
